# 03 — Sampling: temperature, top_p, top_k

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase02/notebooks/03_sampling_params.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objetivo.** Tocar los parámetros de sampling y ver cómo cambia el output. La idea es que después puedas elegirlos a conciencia para tu caso de uso.

**Requisitos.** API key de Groq en `GROQ_API_KEY`.


In [1]:
%pip install --quiet groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.5 MB/s eta 0:00:00


In [2]:
import os
from groq import Groq

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    assert os.environ.get("GROQ_API_KEY"), "Exportá GROQ_API_KEY."

client = Groq()
MODELS = {
    "llama_fast":   "llama-3.1-8b-instant",
    "llama_strong": "llama-3.3-70b-versatile",
    "qwen_reason":  "qwen/qwen3-32b",
    "deepseek":     "deepseek-r1-distill-llama-70b",
    "gemma":        "gemma2-9b-it",
}
MODEL = MODELS["llama_strong"]  # cambiá la clave para probar otros modelos

def generar(prompt, **kwargs):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        **kwargs,
    )
    return resp.choices[0].message.content


## 1. `temperature` — el termostato de la creatividad

Mismo prompt, distintas temperaturas. Observá la diferencia de tono y vocabulario.


In [3]:
PROMPT = "Escribime un poema corto (4 versos) sobre el otoño en Buenos Aires."

for temp in [0.0, 0.3, 0.7, 1.2]:
    print(f"--- temperature = {temp} ---")
    print(generar(PROMPT, temperature=temp))
    print()


--- temperature = 0.0 ---
En Buenos Aires, el otoño llega suave,
Con vientos frescos y un sol que se esconde.
Las hojas caen, doradas y rojizas, 
Y la ciudad se viste de un manto de oro.

--- temperature = 0.3 ---
En Buenos Aires, el otoño llega suave,
Con vientos frescos y un sol que se esconde.
Las hojas caen, doradas y rojizas, 
Y la ciudad se viste de un manto de oro.

--- temperature = 0.7 ---
En Buenos Aires, el otoño llega,
Con sus colores y su suave brisa.
Las hojas caen, doradas y rojas,
Y la ciudad se viste de melancolía.

--- temperature = 1.2 ---
En Buenos Aires, el otoño llega,
con vientos frescos y días cálidos stilla.
Las hojas caen, formando un manto dorado,
en la ciudad que late, con un ritmo relajado.



- `temperature=0` → el modelo elige siempre el token más probable. Determinista.
- Valores altos → distribución más plana, salidas diversas (y a veces incoherentes).


## 2. Reproducibilidad: el bug del "siempre lo mismo"

Con temperatura baja, dos llamadas seguidas dan respuestas casi idénticas. Útil cuando necesitás determinismo (testing, código).


In [4]:
PROMPT = "Listame 3 razones por las que se prefiere PostgreSQL sobre MySQL."

for i in range(2):
    print(f"--- Llamada {i+1} (temperature=0) ---")
    print(generar(PROMPT, temperature=0))
    print()


--- Llamada 1 (temperature=0) ---
¡Claro! A continuación, te presento 3 razones por las que se prefiere PostgreSQL sobre MySQL:

1. **Soporte para tipos de datos avanzados**: PostgreSQL ofrece un conjunto más amplio de tipos de datos, incluyendo tipos de datos espaciales, temporales y de arrays, lo que lo hace más versátil y flexible para manejar datos complejos. Por ejemplo, PostgreSQL admite tipos de datos como `geometry`, `geography` y `interval`, que no están disponibles en MySQL.

2. **Soporte para transacciones y concurrencia**: PostgreSQL tiene un modelo de concurrencia más avanzado que MySQL, lo que significa que puede manejar transacciones y concurrencia de manera más eficiente. Esto se debe a que PostgreSQL utiliza un modelo de concurrencia multiversionado, que permite que varias transacciones se ejecuten simultáneamente sin bloquear el acceso a los datos. En cambio, MySQL utiliza un modelo de concurrencia basado en bloqueos, que puede generar problemas de concurrencia y redu

## 3. `top_p` — nucleus sampling

Muestrea solo del conjunto de tokens cuya probabilidad acumulada llega a P. Recorta la "cola larga".


In [6]:
PROMPT = "Inventame el nombre de una banda de rock progresivo argentina."

for p in [0.0]:
    print(f"--- top_p = {p} ---")
    for _ in range(1):
        print("  ·", generar(PROMPT, temperature=0.0, top_p=p))
    print()


--- top_p = 0.0 ---
  · ¡Claro! Me alegra inventar un nombre para una banda de rock progresivo argentina. Aquí te dejo algunas opciones:

1. **Kaos Cósmico**: Un nombre que refleja la complejidad y la experimentación característica del rock progresivo, con un toque de misterio y aventura.
2. **Sueños de Fuego**: Un nombre que evoca la pasión y la energía del rock argentino, con un toque de poesía y romanticismo.
3. **Ciclos**: Un nombre que sugiere la idea de ciclos y patrones en la música, con un toque de complejidad y profundidad.
4. **La Marea**: Un nombre que refleja la idea de un movimiento constante y en evolución, con un toque de poder y fuerza.
5. **Eclipse**: Un nombre que sugiere la idea de un momento de cambio y transformación, con un toque de misterio y suspense.

Pero si tuviera que elegir uno solo, te propongo: **Kaos Cósmico**. Me parece un nombre que refleja bien la esencia del rock progresivo argentino, con un toque de originalidad y creatividad.

¿Te gusta? ¿Quieres q

## Cuándo usar qué

| Caso | temperature | top_p |
|---|---|---|
| Código, factual, extracción | 0.0 – 0.3 | 1.0 |
| Conversación natural | 0.6 – 0.8 | 0.9 |
| Creatividad, brainstorming | 0.9 – 1.2 | 0.95 |
| Determinismo (tests) | 0.0 | 1.0 |
